# Análisis de Datos con Python

Librería Pandas

In [1]:
import pandas as pd

In [2]:
# Importar base de datos de exoplanetas
data_url = "https://exoplanetarchive.ipac.caltech.edu/cgi-bin/nstedAPI/nph-nstedAPI?table=cumulative&select=kepid,kepoi_name,koi_disposition,koi_period,koi_impact,koi_duration,koi_depth,koi_prad,koi_teq,koi_insol,koi_model_snr&format=csv"
df_0 = pd.read_csv(data_url)

In [3]:
type(df_0)

pandas.core.frame.DataFrame

In [5]:
# Visualización de una pequeña porción de la base de datos
df_0.head()

,kepid,kepoi_name,koi_disposition,koi_period,koi_impact,koi_duration,koi_depth,koi_prad,koi_teq,koi_insol,koi_model_snr
0,10797460,K00752.01,CONFIRMED,9.488036,0.146,2.95750,615.8,2.26,793.0,93.59,35.8
1,10797460,K00752.02,CONFIRMED,54.418383,0.586,4.50700,874.8,2.83,443.0,9.11,25.8
2,10811496,K00753.01,CANDIDATE,19.899140,0.969,1.78220,10829.0,14.60,638.0,39.30,76.3
3,10848459,K00754.01,FALSE POSITIVE,1.736952,1.276,2.40641,8079.2,33.46,1395.0,891.96,505.6
4,10854555,K00755.01,CONFIRMED,2.525592,0.701,1.65450,603.3,2.75,1406.0,926.16,40.9


In [8]:
# Dimensiones de la base
df_0.shape

(9564, 11)

In [9]:
# Nombre de las columnas
df_0.columns

Index(['kepid', 'kepoi_name', 'koi_disposition', 'koi_period', 'koi_impact',
       'koi_duration', 'koi_depth', 'koi_prad', 'koi_teq', 'koi_insol',
       'koi_model_snr'],
      dtype='object')

In [11]:
# Tipo de datos
df_0.dtypes

,0
kepid,int64
kepoi_name,object
koi_disposition,object
koi_period,float64
koi_impact,float64
koi_duration,float64
koi_depth,float64
koi_prad,float64
koi_teq,float64
koi_insol,float64


Limpieza de la base de datos
1.- Eliminar los renglones que contengan valores `NaN` (Not a number)

In [13]:
# Identificación del número de renglones por columna que contienen valores NaN
nans_por_col=df_0.isna().sum(axis=0)
nans_por_col

,0
kepid,0
kepoi_name,0
koi_disposition,0
koi_period,0
koi_impact,363
koi_duration,0
koi_depth,363
koi_prad,363
koi_teq,363
koi_insol,321


In [14]:
# Eliminación de los renglones que contienen valores NaN
df_1=df_0.dropna().reset_index(drop=True)
df_1

,kepid,kepoi_name,koi_disposition,koi_period,koi_impact,koi_duration,koi_depth,koi_prad,koi_teq,koi_insol,koi_model_snr
0,10797460,K00752.01,CONFIRMED,9.488036,0.146,2.95750,615.8,2.26,793.0,93.59,35.8
1,10797460,K00752.02,CONFIRMED,54.418383,0.586,4.50700,874.8,2.83,443.0,9.11,25.8
2,10811496,K00753.01,CANDIDATE,19.899140,0.969,1.78220,10829.0,14.60,638.0,39.30,76.3
3,10848459,K00754.01,FALSE POSITIVE,1.736952,1.276,2.40641,8079.2,33.46,1395.0,891.96,505.6
4,10854555,K00755.01,CONFIRMED,2.525592,0.701,1.65450,603.3,2.75,1406.0,926.16,40.9
...,...,...,...,...,...,...,...,...,...,...,...
9196,10090151,K07985.01,FALSE POSITIVE,0.527699,1.252,3.22210,1579.2,29.35,2088.0,4500.53,453.3
9197,10128825,K07986.01,CANDIDATE,1.739849,0.043,3.11400,48.5,0.72,1608.0,1585.81,10.6
9198,10147276,K07987.01,FALSE POSITIVE,0.681402,0.147,0.86500,103.6,1.07,2218.0,5713.41,12.3
9199,10155286,K07988.01,CANDIDATE,333.486169,0.214,3.19900,639.1,19.30,557.0,22.68,14.0


**Estimados de Locación, Variabilidad y Sesgo**

Locación:
-  Media aritmética: $\overline{x}=\frac{\sum_{i=1}^{N} x_i}{N}$
-  Media recortada: $\overline{x}_t=\frac{\sum_{i=p+1}^{N-p} x_i}{N-2p}$
-  Mediana: Es el valor central de la distribución de datos ordenados de menor a mayor
-  Moda: Es el valor que más se repite en la distribución

In [15]:
# En el siguiente análisis nos enfocaremos en el radio planetario
p_rad=df_1['koi_prad']
p_rad

,koi_prad
0,2.26
1,2.83
2,14.60
3,33.46
4,2.75
...,...
9196,29.35
9197,0.72
9198,1.07
9199,19.30


In [16]:
print(f'El promedio del radio planetario es {p_rad.mean()}')

El promedio del radio planetario es 102.89177806760135


In [17]:
print(f'La mediana del radio planetario es {p_rad.median()}')

La mediana del radio planetario es 2.39


In [19]:
from scipy.stats import trim_mean
print(f'La media truncada con cortes de 20% del radio planetario es {trim_mean(p_rad,0.2)}')

La media truncada con cortes de 20% del radio planetario es 4.664966491577612


Estimados de Variabilidad:
- Desviación Estándar: $s=\sqrt{\frac{\sum_{i=1}^{N} (x_i-\overline{x})²}{N-1}}$
- Rango: diferencia entre los valore máximo y mínimo
- Percentil: Se ordenan los valores de mayor a menor y se selecciona el porcentaje $P$ inicial, el porcentaje $P$ siguiente, etc.
- Rango Intercuartil: diferencia entre el percentil 75 y el 25

In [20]:
print(f'La desviación estándar del radio planetario es {p_rad.std()}')

La desviación estándar del radio planetario es 3077.639126222184


In [21]:
print(f'El Rango intercuartil del radio planetario es {p_rad.quantile(0.75)-p_rad.quantile(0.25)}')

El Rango intercuartil del radio planetario es 13.53


Reto: Hacer el mismo ejercicio para el Período, en la columna `koi_period`.